# SurgeFlow Realtime and Hotlist Monitor

This beginner notebook connects to the SurgeFlow Public API v1 with your generated API key, then displays the two main beta offers:

- Realtime turnover table: the live market board used by the Google Sheets add-on.
- Momentum hotlist: ranked movers with projected turnover and xYest context.

You only need the API key from the SurgeFlow Extension page. The notebook never saves the key to disk.


## 1. Get Your Free API Key

1. Open https://surgeflows.capital/extension.
2. Click the API key request panel.
3. Enter name, email, and agree to the terms.
4. Copy the key that starts with `sf_live_...` when it appears.

Keep the key private. If you share a screenshot of this notebook, clear the output of the key cell first.


In [ ]:
# If your notebook environment does not already have these packages, run once:
# %pip install requests pandas

from __future__ import annotations

import os
import time
import getpass
from datetime import datetime, timezone

import requests
import pandas as pd
from IPython.display import HTML, clear_output, display

BASE_URL = "https://stock-api-c4qdowjxva-uc.a.run.app"
MARKET = "us"          # Choose: us, cn, jp, hk, uk, or in
LIMIT = 25             # Number of realtime rows to show
REFRESH_SECONDS = 60   # The public API monitor refresh interval

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)


## 2. Add Your API Key

The easiest path is to paste the key when prompted. If you prefer environment variables, set `SURGEFLOW_API_KEY` before opening Jupyter.


In [ ]:
api_key = os.getenv("SURGEFLOW_API_KEY", "").strip()

if not api_key:
    api_key = getpass.getpass("Paste your SurgeFlow API key (sf_live_...): " ).strip()

if not api_key.startswith("sf_live_"):
    raise ValueError("Expected a SurgeFlow key that starts with sf_live_.")

headers = {"Authorization": f"Bearer {api_key}"}
print("API key loaded. The secret is hidden and will not be printed.")


## 3. Small Helper Functions

These helpers turn API JSON into normal tables. You can change `MARKET`, `LIMIT`, or `REFRESH_SECONDS` in the setup cell and run the notebook again.


In [ ]:
def call_api(path: str, params: dict | None = None) -> dict:
    """Call a SurgeFlow API endpoint and return JSON."""
    url = f"{BASE_URL}{path}"
    response = requests.get(url, headers=headers, params=params or {}, timeout=30)
    try:
        payload = response.json()
    except ValueError as exc:
        raise RuntimeError(f"API returned non-JSON response: HTTP {response.status_code}") from exc

    if not response.ok or payload.get("ok") is False:
        error = payload.get("error") or {}
        code = error.get("code", f"HTTP_{response.status_code}")
        message = error.get("message", str(payload)[:300])
        raise RuntimeError(f"{code}: {message}")

    return payload


def fetch_realtime(market: str = MARKET, limit: int = LIMIT) -> tuple[dict, pd.DataFrame]:
    payload = call_api(f"/api/v1/markets/{market}/realtime", {"limit": limit})
    data = payload.get("data") or {}
    rows = data.get("rows") or []
    return data, pd.DataFrame(rows)


def fetch_hotlist(market: str = MARKET) -> tuple[dict, pd.DataFrame]:
    payload = call_api(f"/api/v1/markets/{market}/hotlist")
    data = payload.get("data") or {}
    rows = data.get("rows") or []
    return data, pd.DataFrame(rows)


def select_columns(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    existing = [column for column in columns if column in df.columns]
    if not existing:
        return df
    return df.loc[:, existing]


def shorten_company_names(df: pd.DataFrame) -> pd.DataFrame:
    if "company_name" in df.columns:
        df = df.copy()
        df["company_name"] = df["company_name"].astype(str).str.slice(0, 38)
    return df


def display_api_table(title: str, data: dict, df: pd.DataFrame, columns: list[str], limit: int | None = None):
    market = data.get("market", MARKET)
    status = data.get("market_status", "UNKNOWN")
    as_of = data.get("as_of_local") or data.get("as_of_utc") or "unknown"
    quality = data.get("data_quality", "unknown")
    count = data.get("count", len(df))

    display(HTML(
        f"<h3 style='margin-bottom:0.2rem'>{title}</h3>"
        f"<p style='margin-top:0;color:#555'>Market: <b>{market.upper()}</b> | "
        f"Status: <b>{status}</b> | Quality: <b>{quality}</b> | Rows: <b>{count}</b> | As of: {as_of}</p>"
    ))

    if df.empty:
        display(HTML("<p>No rows returned for this market right now.</p>"))
        return

    table = shorten_company_names(select_columns(df, columns))
    if limit is not None:
        table = table.head(limit)
    display(table)


REALTIME_COLUMNS = [
    "rank",
    "ticker",
    "company_name",
    "price",
    "intraday_return_pct",
    "turnover_per_second",
    "accumulated_turnover",
    "projected_turnover",
    "projected_vs_yesterday",
    "previous_day_turnover",
]

HOTLIST_COLUMNS = [
    "hotlist_rank",
    "market",
    "ticker",
    "company_name",
    "industry",
    "factor_style",
    "price",
    "intraday_return_pct",
    "turnover_per_second",
    "market_cap_usd",
    "projected_turnover_usd",
    "projected_vs_yesterday",
    "previous_day_turnover_usd",
]


## 4. Confirm Your Key Works

This call does not consume market data. It confirms the key prefix, member id, plan, scopes, and remaining rate-limit headers.


In [ ]:
me = call_api("/api/v1/me")
me


## 5. One-Time Snapshot

Run this first so you can see the table shapes before starting the refresh loop.


In [ ]:
realtime_data, realtime_df = fetch_realtime(MARKET, LIMIT)
hotlist_data, hotlist_df = fetch_hotlist(MARKET)

display_api_table("Realtime Turnover Table", realtime_data, realtime_df, REALTIME_COLUMNS, limit=LIMIT)
display_api_table("Momentum Hotlist", hotlist_data, hotlist_df, HOTLIST_COLUMNS, limit=25)


## 6. 60-Second Refresh Monitor

The next function refreshes both tables every 60 seconds. By default the final cell runs three cycles, which takes about three minutes. Change `CYCLES = None` for an always-on monitor and stop the notebook cell when you are done.


In [ ]:
def run_monitor(
    market: str = MARKET,
    limit: int = LIMIT,
    refresh_seconds: int = REFRESH_SECONDS,
    cycles: int | None = 3,
):
    cycle = 0
    while True:
        cycle += 1
        clear_output(wait=True)
        now = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
        display(HTML(f"<h2>SurgeFlow Monitor</h2><p>Refresh #{cycle} at {now}</p>"))

        try:
            realtime_data, realtime_df = fetch_realtime(market, limit)
            hotlist_data, hotlist_df = fetch_hotlist(market)

            display_api_table("Realtime Turnover Table", realtime_data, realtime_df, REALTIME_COLUMNS, limit=limit)
            display_api_table("Momentum Hotlist", hotlist_data, hotlist_df, HOTLIST_COLUMNS, limit=25)

        except Exception as exc:
            display(HTML(f"<p style='color:#b00020'><b>API error:</b> {exc}</p>"))
            display(HTML("<p>If this is a rate limit, wait for the reset time shown by the API and try again.</p>"))

        if cycles is not None and cycle >= cycles:
            break

        time.sleep(refresh_seconds)


In [ ]:
# Runs for about three minutes. Change CYCLES to None for an always-on monitor.
CYCLES = 3
run_monitor(market=MARKET, limit=LIMIT, refresh_seconds=60, cycles=CYCLES)


## 7. Next Ideas

- Change `MARKET` to `cn`, `jp`, `hk`, `uk`, or `in`.
- Increase `LIMIT` up to 100 for a broader realtime table.
- Export the latest table to CSV with `realtime_df.to_csv("surgeflow_realtime.csv", index=False)`.
- Keep the free beta limit in mind: 250 requests per day and 50 requests per minute per key.

SurgeFlow is research software only. It is not investment advice or order-routing software.
